# GENERator: A Long-Context Generative Genomic Foundation Model
**ArXivist-generated reproduction notebook**
Paper: https://arxiv.org/abs/2502.07272
Generated: 2026-07-16

This notebook checks the environment, installs the project, explains the model, loads the official
pretrained GENERator (1.2B LLaMA decoder + 6-mer tokenizer), runs a small fine-tuning demo on
synthetic data, and shows the paper's reported results for comparison.

**Use an A100 GPU runtime.** GENERator-1B is 1.2B params (~5GB). On a smaller GPU, set
`load_in_8bit=True` in the model-load cell (needs bitsandbytes).

## 1. Environment check

In [ ]:
import sys, torch
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory/1e9
    print(f"GPU: {name} | VRAM: {vram:.1f} GB")
    if vram < 24:
        print("NOTE: <24GB VRAM. Use load_in_8bit=True for the 1.2B model, or expect OOM.")
else:
    print("WARNING: no GPU. GENERator-1B needs an A100-class GPU.")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 2. Installation

In [ ]:
import subprocess
r = subprocess.run(["pip","install","-q","-e",".."], capture_output=True, text=True)
print(r.stdout[-800:] if r.returncode==0 else r.stderr[-800:])
import transformers; print("transformers:", transformers.__version__)

## 3. Paper overview

**Problem.** Genomic foundation models are constrained by short context, restricted generation, or
huge compute (e.g. Evo2). GENERator targets long-context *generative* modeling efficiently.

**Key ideas.**
- **6-mer tokenization** (Sec 4.2) — best for *autoregressive* DNA (BPE hurts next-token prediction).
- **LLaMA decoder** (Table S12) — 26 layers, d 2048, GQA (32 heads / 4 KV), RoPE, SiLU, ctx 16384 tokens = 98k bp.
- Pretrained on **386B nucleotides** of eukaryotic DNA (functional, gene-centric).

Matches/beats Evo2 on sequence recovery (~19-44x faster), does zero-shot variant effect prediction,
and designs cis-regulatory elements. This repo fine-tunes the official 1.2B weights on classification.

## 4. Component walkthrough: the 6-mer tokenizer

GENERator tokenizes DNA in non-overlapping 6-mers (vocab 4128 = 4096 6-mers + special tokens).

In [ ]:
from src.generator.data.tokenizer import GenomicTokenizer
tok = GenomicTokenizer("GenerTeam/GENERator-eukaryote-1.2b-base", max_len=16)
print("tokenizer:", tok)
enc = tok.encode_batch(["ACGTACGTACGTTTGGCCAA", "TTTTGGGGCCCCAAAATTTT"], max_len=16)
print("input_ids shape:", tuple(enc["input_ids"].shape), "| vocab:", tok.tok.vocab_size)
print("example ids:", enc["input_ids"][0][:8].tolist())

## 5. Load the pretrained GENERator classifier

Loads `GenerTeam/GENERator-eukaryote-1.2b-base` (LLaMA causal LM) and attaches a linear head on the
last-token (`<EOS>`) embedding, per the paper's Appendix C.4. Set `load_in_8bit=True` for smaller GPUs.

In [ ]:
from src.generator.models.classifier import GENERatorClassifier
# On <24GB GPUs, pass load_in_8bit=True (needs: pip install bitsandbytes)
model = GENERatorClassifier.from_pretrained(
    "GenerTeam/GENERator-eukaryote-1.2b-base", num_classes=2,
    device=str(device), load_in_8bit=False)
n = sum(p.numel() for p in model.parameters())
print(model)
print(f"parameter count: {n/1e9:.2f}B")

## Mini-training on synthetic data

No downloads: a tiny GC-rich vs AT-rich binary task. Loss should drop within a few steps.

In [ ]:
import random, torch, torch.nn.functional as F
random.seed(0)
def seq(gc): return "".join(random.choice("GCGCAT" if gc else "ATATGC") for _ in range(60))
seqs   = [seq(i%2==0) for i in range(32)]
labels = [0 if i%2==0 else 1 for i in range(32)]

opt = torch.optim.AdamW(model.parameters(), lr=1e-5, betas=(0.9,0.95))
model.train()
for step in range(6):
    b = seqs[step*4:(step+1)*4] or seqs[:4]
    yb = torch.tensor(labels[step*4:(step+1)*4] or labels[:4]).to(device)
    enc = tok.encode_batch(b, max_len=32)
    ids, mask = enc["input_ids"].to(device), enc["attention_mask"].to(device)
    opt.zero_grad()
    loss = F.cross_entropy(model(ids, mask), yb); loss.backward(); opt.step()
    print(f"step {step} loss {loss.item():.4f}")
print("mini-training complete (loss should trend down)")

## 6. Paper results for comparison

In [ ]:
paper_results = {
    "GenomicBenchmarks/human_nontata_promoters (acc)": 0.958,
    "GenomicBenchmarks/human_vs_worm (acc)":           0.980,
    "GenomicBenchmarks/coding_vs_intergenomic (acc)":  0.963,
    "NT/promoter_all (MCC)":                            0.962,
    "Gener/taxonomic_cls (weighted F1)":               0.999,
    "ClinVar VEP (AUROC / AUPRC)":                      "0.910 / 0.930",
    "Sequence recovery overall (acc)":                  0.515,
}
print("GENERator-1B reported results (paper):")
for k,v in paper_results.items():
    print(f"  {k:48s} {v}")
print("\nTo reproduce: download a task, then run train.py + evaluate.py.")
print("  python data/download.py --benchmark genomic_benchmarks --task human_nontata_promoters")
print("  python train.py --config configs/config.yaml")
print("Then feed your metrics back to ArXivist -> Stage 6 Results Comparator.")

## What to do next

1. **Download data (API):** `python data/download.py --benchmark genomic_benchmarks --task human_nontata_promoters`
2. **Fine-tune (A100):** `python train.py --config configs/config.yaml`
3. **Evaluate:** `python evaluate.py --config configs/config.yaml --checkpoint checkpoints/best.pt`
4. **Zero-shot VEP:** `python vep.py --config configs/config.yaml --sequence ACGT... --pos 3000 --ref A --alt G`
5. **Compare:** paste your metrics back to ArXivist (Stage 6).

**Recipe (Appendix B/C.4):** AdamW β=(0.9,0.95), wd 0.1, reduce-on-plateau + early stop, last-token pooling.